In [1]:
import time
start = time.perf_counter()
import pandas as pd
import polars as pl
import numpy as np
import regex as re
import string
from sklearn.feature_extraction.text import CountVectorizer #TfidfVectorizer,
from scipy.sparse import coo_matrix, hstack
#from scipy.stats import kurtosis, skew
import pickle
import lightgbm as lgbm
#import warnings
#warnings.filterwarnings("ignore")

In [2]:
#train = pd.read_csv("/kaggle/input/persandtypo/Pers_models/trainANDpers.csv")
train = pl.read_csv("/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv")
#train = train[:8000]

In [3]:
# al contains all Correct words in a Set.
al = pickle.load(open("/kaggle/input/lgbm-v36/Correct_Words_Set/Correct_Words_Set.pkl", "rb"))


def remove__(x):
    """Removes multiple spaces"""
    return re.sub("\s{2,100000}", "", x) 


def clean_row_for_split(row):
    """Preprocessing required to split the words"""
    row = row.lower()
    row = re.sub("[<>\|\^\@\*²¹©\$\d\&,\/\.\(\)\[\]\s\&\%\#\!\-\_\=\+\:\?\"';]"," ", row)
    row = re.sub("\s{2,10000}", " ", row)
    row = row.split(" ")
    return row


def Spelling_Errors(row):
    """Returns spelling errors as percentage of total words"""
    num = 0
    nn = len(row)
    for tok in row:
        if tok not in al:
            num+=1

    return num/nn


def Unique_words_prcnt(row):
    """Returns Percentage of unique words"""
    return len(set(row))/len(row)



para_col = [pl.col(["full_text"]).str.split("\n\n").alias("paragraph")] 
train = train.with_columns(para_col)

# Since we have already already splitted the text as paragraphs, Now I will remove all the unnecessary spaces
# In my observation doing this helps source text columns.
train = train.with_columns(pl.col("full_text").map_elements(remove__, return_dtype = str).alias("full_text"))

#def span_length(x):
#    num = 0
#    cnt = 0
#    for sp in x:
#        num+= sp.span()[1] - sp.span()[0] - 2
#        cnt+=1
#        
#    return num, cnt

def span_length(x, gap_width = 10):
    """
    Returns Sum  and Count of all the Source/Quoted text with length > gap width.
    NOTE, quoted text also contains quotes so their length always contains 2 more characters.
    Thats why, while returning I subtracted 2 from length. It is just to make more accurate 
    measurements.
    """
    
    num = 0
    cnt = 0
    for sp in x:
        gap = sp.span()[1] - sp.span()[0] 
        if gap > gap_width:
            num+= gap
            cnt+=1
        
    return num-(cnt*2), cnt

def find_span_length(x):
    """
    Find all the spans of double quoted text and pass it to `span_length` function
    to get total span length and number of spans.
    """
    
    spans = list(re.finditer("\"[\w\s\?\[\]\(\)\-\!\:\;\']+\"", x))
    return (span_length(spans))


def PreprocessText(x):
    x = x.lower()
    x = re.sub("[0-9\&\,\.\_\~\+\-\(\)\*#\$\%@!\?:;`\|\"'\[\]\/\\\]", " ", x)
    x = re.sub("[\s\.]{2,100000}", " ", x)
    return x




def ParaPreprocessing(train):
    
    
    #train = train.with_columns(pl.col("full_text").
     #                         map_elements(CapitalPercentage, return_dtype = float).alias("Capital"))
    train = train.explode("paragraph")
    train = train.with_columns(pl.col("paragraph").map_elements(clean_row_for_split).alias("p_cln"))
    
    #train = train.with_columns(pl.col("full_text").
    #                        map_elements(SpacePeriod, return_dtype = int).alias("SP_er"))

    #train = train.with_columns(pl.col("paragraph").map_elements(
        #lambda x: PreprocessText(x), return_dtype=str).alias("text_cln"))
    
    #train = train.with_columns(pl.col("text_cln").map_elements(spellings).alias("spelling_error_num"))
    
    train = train.with_columns(pl.col("paragraph").map_elements(
        lambda x: len(x), return_dtype=int).alias("paragraph_len"))
    
    train = train.with_columns(pl.col("paragraph").map_elements(
        lambda x: len(x.split(".")), return_dtype=int).alias("paragraph_sent_cnt"))
    
    train = train.with_columns(pl.col("paragraph").map_elements(
        lambda x: len(x.split(" ")), return_dtype=int).alias("paragraph_word_cnt"))
    
    train = train.with_columns(pl.col("p_cln").map_elements(
        Spelling_Errors, return_dtype=float).alias("paragraph_errors"))
    
    #train = train.with_columns(pl.col("p_cln").map_elements(
     #   Unique_words_prcnt, return_dtype=float).alias("unique_words_prcnt"))
    
    
    li = []
    lii = []
    for row in train[:,"full_text"]:
        j,jj = find_span_length(row)
        li.append(j)
        lii.append(jj)
    
    train = train.with_columns(pl.col("paragraph").map_elements(lambda x: len(x), return_dtype = int).alias("plen"))
    
    ## SPAN FEATURES FOR PARAGRAPH
    ## span length
    train = train.with_columns(pl.Series("spans", li).alias("spans"))
    
    # spans count
    train = train.with_columns(pl.Series("spansN", lii).alias("spansN"))
    
    # percentage of spans in a paragraph
    train = train.with_columns((pl.col("spans")/pl.col("plen")).alias("spans%"))
    
    # Average span length
    train = train.with_columns((pl.col("spans")/pl.col("spansN")).alias("spansAvg"))
    
    return train


# feature_eng
paragraph_fea = ['paragraph_len','paragraph_sent_cnt','paragraph_word_cnt',
                 "paragraph_errors"] #, "unique_words_prcnt"


def ParaFeatures(train_tmp):
    aggs = [
        *[pl.col("spans").max().alias("SP_Max")],
        *[pl.col("spans").mean().alias("SP_Mean")],
        *[pl.col("spans").sum().alias("SP_Sum")],
        
        *[pl.col("spansN").max().alias("SPN_Max")],
        *[pl.col("spansN").mean().alias("SPN_Mean")],
       #*[pl.col("spansN").sum().alias("SPN_Sum")],
        
        *[pl.col("spans%").max().alias("SP%_Max")],
        *[pl.col("spans%").mean().alias("SP%_Mean")],
        *[pl.col("spans%").sum().alias("SP%_Sum")],
        
        *[pl.col("spansAvg").max().alias("SPAvg_Max")],
        *[pl.col("spansAvg").mean().alias("SPAvg_Mean")],
        *[pl.col("spansAvg").sum().alias("SPAvg_Sum")],
        
        *[pl.col('paragraph').filter(pl.col('paragraph_len') >= i).count().alias(f"paragraph_>{i}_cnt")
          for i in [i for i in range(0, 650, 50)]],
        *[pl.col('paragraph').filter(pl.col('paragraph_len') <= i).count().alias(f"paragraph_<{i}_cnt")
          for i in [25,49]],
        
        *[pl.col('paragraph').filter(pl.col('paragraph_errors') > i).count().alias(f"paragraph_er_>{i}_cnt")
          for i in [i for i in [0,.005,.01,.02,.03,.05,]]],
      #  
       #*[pl.col('paragraph').filter(pl.col('unique_words_prcnt') >= i).count().alias(f"p_uni_words_>{i}_cnt")
       #  for i in [i for i in [.1,.3,.4,.5,.6,.7]]],
        
        
        *[pl.col(fea).max().alias(f"{fea}_max") for fea in paragraph_fea],
        *[pl.col(fea).mean().alias(f"{fea}_mean") for fea in paragraph_fea],
        *[pl.col(fea).min().alias(f"{fea}_min") for fea in paragraph_fea],
        *[pl.col(fea).sum().alias(f"{fea}_sum") for fea in paragraph_fea if fea != "unique_words_prcnt"],
        *[pl.col(fea).first().alias(f"{fea}_first") for fea in paragraph_fea],
        *[pl.col(fea).last().alias(f"{fea}_last") for fea in paragraph_fea],
        *[pl.col(fea).quantile(0.25).alias(f"{fea}_q1") for fea in paragraph_fea],
        *[pl.col(fea).quantile(0.5).alias(f"{fea}_q2") for fea in paragraph_fea],
        *[pl.col(fea).quantile(0.75).alias(f"{fea}_q3") for fea in paragraph_fea],
        ]
    df = train_tmp.group_by(['essay_id'], maintain_order=True).agg(aggs)
    #exp2 = [*[pl.apply(exprs = ["full_text"], function = CapitalPercentage).alias("CapitalPercentage")]
            
    #print(train_tmp)
    #train_tmp = train_tmp.group_by(["full_text"], maintain_order = True).agg(
    #    pl.col("full_text").map_elements(lambda x: CapitalPercentage(x), return_dtype = float).alias("Capital"))
    #print(train_tmp)
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
        #lambda x: CharPresent(x, ','), return_dtype=int).alias("comma"))
    
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
        #lambda x: CharPresent(x, '\.'), return_dtype=int).alias("period"))
    
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
     #   lambda x: CapitalPercentage(x), return_dtype=float).alias("CapitalPercentage"))
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
    #     CapitalPercentage, return_dtype=float).alias("CapitalPercentage"))
        
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
    #    lambda x: len_ext_pipe(x), return_dtype=int).alias("quotes_cnt"))
    
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
    #    lambda x: topics_at_all(x), return_dtype=int).alias("all_topic_cnt"))
    
    #train_tmp = train_tmp.with_columns(pl.col("full_text").map_elements(
    #    lambda x: topics_at_firstlast(x), return_dtype=int).alias("first_last_topic_cnt")) 
    
    
    df = df.to_pandas()
    #df2 = train_tmp.to_pandas()
    #df = df.join(train_tmp[:, "CapitalPercentage"].to_pandas()) #, "all_topic_cnt", "first_last_topic_cnt"
    #df = df.join(df2["CapitalPercentage"])
    return df, train_tmp

#tr.join(train[:, ["full_text", "score"]].to_pandas())
                     
def SentPreprocessing(train):
    train = train.with_columns(pl.col("full_text").str.split(".").alias("sentence"))
    train = train.explode("sentence")
    train = train.with_columns(pl.col("sentence").map_elements(clean_row_for_split).alias("s_cln"))
    train = train.with_columns(pl.col("sentence").map_elements(
        lambda x: len(x), return_dtype=int).alias("sentence_len"))
    train = train.with_columns(pl.col("sentence").map_elements(
        lambda x: len(x.split(" ")), return_dtype=int).alias("sentence_word_cnt"))
    train = train.with_columns(pl.col("s_cln").map_elements(
        Spelling_Errors, return_dtype=float).alias("sentence_errors"))
    
    train = train.with_columns(pl.col("s_cln").map_elements(
        Unique_words_prcnt, return_dtype=float).alias("s_unique_words_prcnt"))
    
    return train


sent_feats = ["sentence_len", "sentence_word_cnt", "sentence_errors", "s_unique_words_prcnt"]
def SentFeatures(train):
    
    aggs = [
        *[pl.col(["sentence"]).filter(pl.col("sentence_len")>=i).count().alias(f"sent_len>{i}") 
                                    for i in range(0, 160, 10)],
        *[pl.col(["sentence"]).filter(pl.col("sentence_len")<=i).count().alias(f"sent_len<{i}") 
                                    for i in [15,50]], #range(15, 160, 30)],#
        
        *[pl.col('sentence').filter(pl.col('sentence_errors') > i).count().alias(f"sentence_er_>{i}_cnt")
          for i in [i for i in [.01,.02,.03,.05,]]],
        
        #*[pl.col('sentence').filter(pl.col('s_unique_words_prcnt') > i).count().alias(f"s_words_prcnt_>{i}_cnt")
        #  for i in [i for i in [.01,.02,.03,.4,.05,]]],
        
        *[pl.col(fea).max().alias(f"{fea}_max") for fea in sent_feats],
        *[pl.col(fea).mean().alias(f"{fea}_mean") for fea in sent_feats],
        *[pl.col(fea).min().alias(f"{fea}_min") for fea in sent_feats],
        *[pl.col(fea).sum().alias(f"{fea}_sum") for fea in sent_feats if fea != "s_unique_words_prcnt"],
        *[pl.col(fea).first().alias(f"{fea}_first") for fea in sent_feats],
        *[pl.col(fea).last().alias(f"{fea}_last") for fea in sent_feats],
        *[pl.col(fea).quantile(0.25).alias(f"{fea}_q1") for fea in sent_feats],
        *[pl.col(fea).quantile(0.5).alias(f"{fea}_q2") for fea in sent_feats],
        *[pl.col(fea).quantile(0.75).alias(f"{fea}_q3") for fea in sent_feats],
        ]
    
    train = train.group_by(["essay_id"], maintain_order = True).agg(aggs)
    df = train.to_pandas()
    return df


def WordPreprocessing(train):
    train = train.with_columns(pl.col("full_text").str.split(" ").alias("word"))
    train = train.explode("word")
    train = train.with_columns(pl.col("word").map_elements(
        lambda x: len(x), return_dtype=int).alias("word_len"))
    train = train.filter(pl.col("word_len")>0)
    return train


def WordFeatures(train):
    aggs = [
        *[pl.col(["word"]).filter(pl.col("word_len")>=i).count().alias(f"word_len>{i}") 
                                    for i in range(1, 16)],
        *[pl.col("word_len").max().alias("word_len_max")],
        *[pl.col("word_len").mean().alias("word_len_mean")],
        *[pl.col("word_len").min().alias("word_len_min")],
        #*[pl.col(fea).kurtosis().alias(f"{fea}_kurtosis") for fea in paragraph_fea],
        *[pl.col("word_len").quantile(0.25).alias("word_len_q1")],
        *[pl.col("word_len").quantile(0.5).alias("word_len_q2")],
        *[pl.col("word_len").quantile(0.75).alias("word_len_q3")]
        ]
    
    train = train.group_by(["essay_id"], maintain_order = True).agg(aggs)
    df = train.to_pandas()
    return df


# Paragraph Features
para_df = ParaPreprocessing(train)
para_df, cc = ParaFeatures(para_df)

# Sentence Features
sent_df = SentPreprocessing(train)
sent_df = SentFeatures(sent_df)

# Word Features
word_df = WordPreprocessing(train)
word_df = WordFeatures(word_df)
 

def get_feats(para= para_df, sent=sent_df, word=word_df):
    """Returns all Newly created Features"""
    feats = pd.merge(para_df, sent_df, how = "inner", on = "essay_id")
    feats = pd.merge(feats, word_df, how = "inner", on = "essay_id")
    return feats
    
feats = get_feats()

/tmp/ipykernel_17/304893634.py:95: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  train = train.with_columns(pl.col("paragraph").map_elements(clean_row_for_split).alias("p_cln"))
/tmp/ipykernel_17/304893634.py:230: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  train = train.with_columns(pl.col("sentence").map_elements(clean_row_for_split).alias("s_cln"))


In [4]:
li = []
lii = []
for row in train[:,"full_text"]:
    j,jj = find_span_length(row)
    li.append(j)
    lii.append(jj)

In [5]:
## SPAN FEATURES FOR FULL_TEXT 
train = train.with_columns(pl.col("full_text").map_elements(lambda x: len(x)).alias("length"))
train = train.with_columns(pl.Series("spans", li).alias("spans"))
train = train.with_columns(pl.Series("spansN", lii).alias("spansN"))
train = train.with_columns((pl.col("spans")/ pl.col("length")).alias("span%"))
train = train.with_columns((pl.col("spans")/ pl.col("spansN")).alias("spansAvg"))

/tmp/ipykernel_17/840663712.py:2: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  train = train.with_columns(pl.col("full_text").map_elements(lambda x: len(x)).alias("length"))


In [6]:
# Loading Vectorizers
tf_cnt_n = pickle.load(open("/kaggle/input/lgbm-v42/cpu_lgbmV42/count_tokenizer.pkl", "rb"))
tf_cnt_n2 = pickle.load(open("/kaggle/input/lgbm-v42/cpu_lgbmV42/count_tokenizer2.pkl", "rb"))

tok_tff = tf_cnt_n.transform([i for i in train[:,"full_text"]])
tok_tf = tf_cnt_n2.transform([i for i in train[:,"full_text"]])

/opt/conda/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator CountVectorizer from version 1.4.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# Horizontally stacking all the features
Feats = hstack((coo_matrix(feats.drop(columns = ["essay_id"], axis = 1)),
                coo_matrix(tok_tf),
                coo_matrix(tok_tff),
                coo_matrix(train[:,["spans", "span%", "spansAvg"]]),
                
               )) 

Feats = pd.DataFrame(Feats.toarray())

# Feature selection for LGBM Classifier
FeatsC = Feats.iloc[:, pickle.load(open("/kaggle/input/lgbm-v42/cpu_lgbmV42/all_cls_imps.pkl", "rb"))].copy()
#Feats = Feats[pickle.load(open("/kaggle/input/cpu-lgbmv9/cpu_lgbmV9/all_impsV2.pkl", "rb"))]

In [8]:
def quadratic_weighted_kappa(y_true, y_pred):

        # For lgb
        y_true = y_true + a
        y_true = y_true.round().clip(1,6)
        y_pred = (y_pred + a).clip(1,6).round()


        y_true = y_true.round()
        y_pred = y_pred.round()
        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        return 'QWK', qwk, True


def qwk_obj(y_true, y_pred):
    labels = y_true + a
    preds = y_pred + a
    preds= preds.clip(1,6)
    f = 1/2*np.sum((preds-labels)**2)
    g = 1/2*np.sum((preds-a)**2+b)
    df = preds - labels 
    dg = preds - a
    grad = (df/g - f*dg/g**2)*len(labels)
    hess = np.ones(len(labels))
    return grad, hess
a = 2.998
b = 1.092

def quadratic_weighted_kappa_cls(y_true, y_pred):
        y_pred = y_pred.argmax(axis = -1)
    
        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        return 'QWK', qwk, True

In [9]:
# Getting predictions from LGBM Classifier
predictions = np.zeros((5,len(train),6))
for i in range(5):
    mpath = f"/kaggle/input/lgbm-v42/cpu_lgbmV42/clsV1_{i}.pkl"
    mod = pickle.load(open(mpath, "rb"))
    preds = mod.predict_proba(FeatsC)
    predictions[i] = preds

/opt/conda/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.4.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.4.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.4.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more inf

In [10]:
predictions = np.mean(predictions, axis = 0)

# Adding PROBABILITIES from LGBM Classifier to the dataset for LGBM Regressor
Feats = Feats.join(pd.DataFrame(predictions, columns = [f"{i}_" for i in range(6)]))

# Feature selection for LGBM Regressor
Feats = Feats.loc[:, pickle.load(open("/kaggle/input/lgbm-v42/cpu_lgbmV42/all_reg_imps.pkl", "rb"))]

In [11]:
# Getting predictions from LGBM Regressor
predictions = np.zeros((5,len(train)))
for i in range(5):
    mpath = f"/kaggle/input/lgbm-v42/cpu_lgbmV42/regV1_{i}.pkl"
    mod = pickle.load(open(mpath, "rb"))
    preds = mod.predict(Feats)+a
    predictions[i] = preds

In [12]:
predictions = np.mean(predictions,axis = 0)

#thresh = 
# [1.5, 2.56, 3.452, 4.502, 5.233]

def cut(x):
    if x<1.5:
        return 1 
    elif x<2.56:
        return 2 
    elif x<3.452:
        return 3 
    elif x<4.502:
        return 4 
    elif x<5.233:
        return 5 
    else:
        return 6

sub= pd.DataFrame({"essay_id": train[:, "essay_id"]})
sub["score"] = list(map(cut, predictions))

In [13]:
sub= pd.DataFrame({"essay_id": train[:, "essay_id"]})
sub["score"] = predictions.clip(1,6).round().astype(int)
#predictions.clip(1,6).round().astype(int)#list(map(cut, predictions))

In [14]:
sub.to_csv("submission.csv", index = False)